# ShowPtycho: Apple MPS — basic — metadata calibration

Load your local H5/QEM data, browse with Show4DSTEM, fit SSB, then review
and save the calibration. No aberration values are required.

Install the editable bobleesj GPU and Widget forks using the
[setup guide](setup/README.md). No data are downloaded.

In [ ]:
from pathlib import Path

from quantem.gpu import SSB, detector, io, screening
import quantem.widget as qw

qw.profile()
device = "mps"
source = Path("scan_master.h5")  # Your local acquisition.
save_dir = "ssb-results-mps"



## Check acquisition suitability

The default tilt/depth fit requires measured spatial frequencies in its fitting band (0.2–0.9 Å⁻¹). Coarsely sampled scans may not reach that band even when metadata are complete. The fitter stops with an explanation; Auto upsampling cannot recover missing measured frequencies.

For such an acquisition, explicitly replace the fitting call with ordinary SSB:

```python
fit = ssb.find_aberrations(
    search="primary", tilt=False, objective="phase_variance",
)
```

This does not fit tilt/depth or broad higher-order aberrations. In the advanced notebook, skip tilt/depth landscapes and the four-case joint-tilt comparison for this alternative.

## Load data

Keep the master, detector shards and matching microscope metadata together. Recorded calibration is used without manual overrides. Missing voltage or scan sampling must be supplied from an independent calibration; use the manual notebook if necessary.

In [ ]:
data = io.load(source, device=device)
data.calibration

## Browse with Show4DSTEM

Move across the scan to inspect diffraction patterns. Drag the virtual detector
or choose BF/ADF presets to inspect specimen regions. ShowPtycho below reuses
this same loaded acquisition; browsing does not crop or bin its data.

In [ ]:
browse = qw.Show4DSTEM(data)
browse

## Use acquisition calibration

Recorded detector angular sampling is retained. If the convergence semiangle is missing, screening estimates it from the measured bright-field disk and recorded angular sampling. This depends on the detector calibration being correct. If both are missing, use the manual notebook. The following output identifies the estimate when used.

In [ ]:
products = screening.prepare(data)
products.metadata.get("semiangle_mrad_source", "Check the recorded semiangle in the calibration table")

## Inspect the mean diffraction pattern

Screening reuses the loaded acquisition and provides BF, DF, ADF and mean diffraction arrays. Inspect the disk before fitting.

In [ ]:
detector.show(data, size="medium", columns=2)


For individual arrays or custom detector angles, see the advanced tutorial.


## Fit and reconstruct automatically

The default uses **200 TPE trials for the initial optics, tilt and depth fit**,
then jointly refines all aberrations through fifth order with Newton.
`search="broad"` adds 400 fifth-order TPE trials before the same refinement.

Fitting uses native scan sampling and all valid bright-field pixels.
**Auto** sets the displayed reconstruction sampling from the calibrated bandwidth.
A supplied rotation is retained; estimated rotation is reviewed in the initial fit.
Phase polarity alone does not independently prove the rotation.

More fitted terms do not establish resolution or physical probe aberrations.
Newton can stop early; reaching its evaluation limit does not prove convergence.


The initial phase appears while fifth-order refinement continues; the same view updates when finished. Use `show=False` to fit without a notebook preview.


In [ ]:
ssb = SSB(
    data, backend="mps",
    semiangle_mrad=products.metadata.get("semiangle_mrad"),
)
fit = ssb.find_aberrations()
result = ssb.reconstruct(fit, upsampling_factor="auto")
result.report()

## Review

Adjust C10, astigmatism or tilt in the widget and inspect the phase and FFT.
Auto sampling is also used by the viewer. Display contrast and phase negation
change only the view.
Auto stops at **4×**. To request 8× explicitly, use
`ssb.show(show_fft=True, save_dir=save_dir, upsampling=8)` or the
Upsampling menu. Settings show when nominal support needs more than 4×.


In [ ]:
viewer = ssb.show(show_fft=True, save_dir=save_dir)
viewer

## Save the reviewed calibration

Run this after finishing your adjustments. Reopen the same acquisition with the
same `save_dir` to restore the saved model in ShowPtycho. Use a separate output
folder for each acquisition.

In [ ]:
viewer.save_calibration()

## Optional independent calibration

To replace the detector calibration, use the manual notebook: supply a known convergence semiangle and fit the probe disk. A disk radius alone cannot establish an angular scale. Do not estimate a semiangle from metadata and then use that same estimate as an independent recalibration.